In [6]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn as nn
import torch.optim as optim
import logging
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import random

In [10]:
# Configuração de Logging
logging.basicConfig(filename='process_log.log', level=logging.ERROR,
                    format='%(asctime)s %(levelname)s %(message)s')

class ECGDataset(Dataset):
    def __init__(self, base_dir):
        self.ecg_data = []
        self.labels = []
        self.prontuarios = []
        self.load_processed_beats(base_dir)
    
    def load_processed_beats(self, base_dir):
        for label in ['0.0', '3.0']:
            folder_path = os.path.join(base_dir, label)
            for prontuario_folder in os.listdir(folder_path):
                prontuario_path = os.path.join(folder_path, prontuario_folder)
                if os.path.isdir(prontuario_path):
                    for filename in os.listdir(prontuario_path):
                        if filename.endswith('.npy'):
                            file_path = os.path.join(prontuario_path, filename)
                            try:
                                data = np.load(file_path)
                                if data.size == 0:
                                    raise ValueError(f"Arquivo vazio: {file_path}")
                                self.ecg_data.append(data)
                                label_value = int(label.split('.')[0])
                                self.labels.append(0 if label_value == 0 else 1)  # Ajuste aqui
                                self.prontuarios.append(prontuario_folder)
                            except Exception as e:
                                logging.error(f"Erro ao carregar o arquivo {file_path}: {e}")
    
    def __len__(self):
        return len(self.ecg_data)
    
    def __getitem__(self, idx):
        signal = torch.tensor(self.ecg_data[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        prontuario = self.prontuarios[idx]
        return signal, label, prontuario


# Definir diretório base
base_dir = "/home/leo-vitor/Documentos/UFC/biosinais/codigos/Dist_Cond_AtrioVent_Classification/Transformer-based_detection/processed_beats"

# Criar o dataset
dataset = ECGDataset(base_dir)

# Verificar rótulos no dataset
unique_labels = set(dataset.labels)
print(f"Unique labels in the dataset: {unique_labels}")

# Ajustar rótulos, se necessário
corrected_labels = [0 if label == 0 else 1 for label in dataset.labels]
dataset.labels = corrected_labels

# Amostra aleatória para determinar a dimensão de entrada
sample_signal, _, _ = dataset[0]
input_dim = sample_signal.size(0)  # Tamanho do sinal de ECG

# Verificar o número de elementos no dataset
print(f"Number of loaded ECG data: {len(dataset)}")
print(f"Number of labels: {len(dataset.labels)}")

Unique labels in the dataset: {0, 1}
Number of loaded ECG data: 3148660
Number of labels: 3148660


In [7]:
# Obter listas de índices para cada prontuário
prontuario_to_indices = {}
for idx, prontuario in enumerate(dataset.prontuarios):
    if prontuario not in prontuario_to_indices:
        prontuario_to_indices[prontuario] = []
    prontuario_to_indices[prontuario].append(idx)

# Dividir os prontuários em treino, validação e teste
train_prontuarios, temp_prontuarios = train_test_split(list(prontuario_to_indices.keys()), test_size=0.3, random_state=42)
val_prontuarios, test_prontuarios = train_test_split(temp_prontuarios, test_size=0.5, random_state=42)

# Converter listas de listas em listas simples de índices
train_indices = [idx for prontuario in train_prontuarios for idx in prontuario_to_indices[prontuario]]
val_indices = [idx for prontuario in val_prontuarios for idx in prontuario_to_indices[prontuario]]
test_indices = [idx for prontuario in test_prontuarios for idx in prontuario_to_indices[prontuario]]

# Criar subsets para cada conjunto
train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)
test_dataset = Subset(dataset, test_indices)

# Criar DataLoaders para treinamento, validação e teste
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [8]:
# Definindo parâmetros do modelo
input_dim = 77  
embed_dim = 32
nhead = 8
nhid = 512
nlayers = 6
num_classes = 2

# Definição do modelo
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=0.1)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

class EmbeddingLayer(nn.Module):
    def __init__(self, input_dim, embed_dim):
        super(EmbeddingLayer, self).__init__()
        self.linear = nn.Linear(input_dim, embed_dim)
        self.dropout = nn.Dropout(p=0.1)
        self.positional_encoding = PositionalEncoding(embed_dim)
    
    def forward(self, x):
        x = self.linear(x)
        x = self.dropout(x)
        x = self.positional_encoding(x)
        return x

class TransformerModel(nn.Module):
    def __init__(self, embed_dim, nhead, nhid, nlayers, num_classes, dropout=0.5):
        super(TransformerModel, self).__init__()
        self.model_type = 'Transformer'
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=nhead, dim_feedforward=nhid, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=nlayers)
        self.decoder = nn.Linear(embed_dim, num_classes)

    def forward(self, src):
        output = self.transformer_encoder(src)
        output = self.decoder(output.mean(dim=0))
        return output

class AnomalyDetectionModel(nn.Module):
    def __init__(self, input_dim, embed_dim, nhead, nhid, nlayers, num_classes):
        super(AnomalyDetectionModel, self).__init__()
        self.embedding = nn.Linear(input_dim, embed_dim)
        self.transformer_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=nhead, dim_feedforward=nhid),
            num_layers=nlayers
        )
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = x.transpose(0, 1)  # Transformer espera entrada (sequence length, batch size, embed dim)
        x = self.transformer_encoder(x)
        x = x.transpose(0, 1)  # Retorne ao formato original (batch size, sequence length, embed dim)
        # Removemos a operação de mean(dim=1) para manter a saída para cada amostra
        x = self.fc(x)
        return x


# Instanciar o modelo
model = AnomalyDetectionModel(input_dim, embed_dim, nhead, nhid, nlayers, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

/home/leo-vitor/anaconda3/lib/python3.11/site-packages/torch/nn/modules/transformer.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [9]:
#Função de treinamento
def train(model, data_loader, criterion, optimizer, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for inputs, labels, _ in data_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            print(f"Forma dos dados de entrada: {inputs.shape}")
            print(f"Forma da saída do modelo: {outputs.shape}")
            print(f"Forma dos rótulos: {labels.shape}")
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(data_loader)
        print(f'Epoch {epoch+1}, Loss: {avg_loss}')

# Treinar o modelo
train(model, train_loader, criterion, optimizer, num_epochs=20)


# Função para checar o tamanho das entradas
def check_input_sizes(dataset, num_samples=10):
    sample_indices = random.sample(range(len(dataset)), num_samples)
    for idx in sample_indices:
        signal, label, prontuario = dataset[idx]
        print(f"Prontuário: {prontuario}, Label: {label}, Length of input: {len(signal)}")

# Checar o tamanho das entradas no conjunto de treino
print("Train Dataset:")
check_input_sizes(train_dataset)

# Checar o tamanho das entradas no conjunto de validação
print("\nValidation Dataset:")
check_input_sizes(val_dataset)

# Checar o tamanho das entradas no conjunto de teste
print("\nTest Dataset:")
check_input_sizes(test_dataset)

TypeError: list indices must be integers or slices, not str

In [ ]:
# Função de avaliação
def evaluate(model, data_loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels, _ in data_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    avg_loss = total_loss / len(data_loader)
    accuracy = correct / total
    return avg_loss, accuracy

# Avaliar o modelo
val_loss, val_accuracy = evaluate(model, val_loader, criterion)
test_loss, test_accuracy = evaluate(model, test_loader, criterion)

print(f'Validation Loss: {val_loss}, Validation Accuracy: {val_accuracy}')
print(f'Test Loss: {test_loss}, Test Accuracy: {test_accuracy}')